# MrBilit Smart Auto-Suggest

A lightweight NLP/search-ranking system that produces five ranked destination
suggestions from a partially typed query.

The solution combines:
- Persian prefix matching,
- English city-name matching,
- Persian/English keyboard-layout conversion,
- Levenshtein edit distance for typos,
- destination popularity as a tie-breaker/fallback.

The submitted solution achieved a **Mean RBO score of 60.53** on the Quera
benchmark, above the full-score threshold of 30.


In [ ]:
import pandas as pd
from Levenshtein import distance


## 1. Load reference and search data


In [ ]:
typo = pd.read_csv("typo_char.csv")
cities = pd.read_csv("iran_cities.csv")
org_data = pd.read_json("mrbilit_search.json")

print("Search data shape:", org_data.shape)
org_data.head()


## 2. Keep ground-transport searches

The exercise focuses on `bus` and `taxi` searches.


In [ ]:
data = org_data[
    org_data["ServiceType"].isin(["bus", "taxi"])
].copy()

data.shape


## 3. Add Persian and English city aliases

`AcceptString` can contain a terminal suffix after `-`. A city-level alias is
created for matching, while the original `AcceptString` is preserved because
the final suggestions must return the original accepted destination labels.


In [ ]:
data["CityFA"] = (
    data["AcceptString"]
    .str.split("-", n=1)
    .str[0]
    .str.strip()
)

city_names = cities[["City FA", "City EN"]]

data = data.merge(
    city_names,
    left_on="CityFA",
    right_on="City FA",
    how="left"
)

data.head()


## 4. Expand historical typing sequences

Each `TypedStrings` list is exploded into individual typed states. This table
is useful for inspecting user typing behavior and feature engineering. The
ranker below remains a lightweight rule-based/probabilistic baseline rather
than a parametric ML model.


In [ ]:
train_data = data.explode("TypedStrings").copy()
train_data = train_data.rename(columns={"TypedStrings": "Typed"})
train_data = train_data.dropna(subset=["Typed"])
train_data = train_data[train_data["Typed"] != ""].copy()

train_data.head()


## 5. Keyboard-layout conversion

`typo_char.csv` maps English keyboard keys to their Persian counterparts.
Both directions are prepared so the ranker can handle users typing with the
wrong active keyboard language.


In [ ]:
en_to_fa = dict(zip(typo["EN"], typo["FA"]))
fa_to_en = {fa: en for en, fa in en_to_fa.items()}

def keyboard_to_fa(text):
    return "".join(
        en_to_fa.get(ch, ch)
        for ch in str(text).lower()
    )

def keyboard_to_en(text):
    return "".join(
        fa_to_en.get(ch, ch)
        for ch in str(text)
    )

train_data["Typed_to_fa"] = train_data["Typed"].apply(keyboard_to_fa)
train_data["Typed_to_en"] = train_data["Typed"].apply(keyboard_to_en)

train_data.head()


## 6. Build candidate destinations

Destination frequency acts as a useful ranking signal when several candidates
match equally well.


In [ ]:
popularity = data["AcceptString"].value_counts()

candidates = (
    data[["AcceptString", "CityFA", "City EN"]]
    .drop_duplicates("AcceptString")
    .copy()
)

candidates["freq"] = candidates["AcceptString"].map(popularity)

candidates["AcceptString"] = candidates["AcceptString"].fillna("")
candidates["CityFA"] = candidates["CityFA"].fillna("")
candidates["City EN"] = candidates["City EN"].fillna("")


## 7. Ranking function

Ranking priority:

1. Direct Persian/original prefix match.
2. English city-name prefix match.
3. English-keyboard-to-Persian conversion.
4. Persian-keyboard-to-English conversion.
5. Levenshtein edit distance, with popularity as a tie-breaker.

The function always returns unique suggestions.


In [ ]:
def suggest(text, k=5):
    text = str(text).strip()
    text_lower = text.lower()

    text_fa = keyboard_to_fa(text)
    text_en = keyboard_to_en(text).lower()

    result = []

    def add_matches(mask):
        matched = candidates[mask].sort_values("freq", ascending=False)

        for value in matched["AcceptString"]:
            if value not in result:
                result.append(value)

            if len(result) == k:
                return True

        return False

    # 1) Direct Persian/original prefix
    if add_matches(
        candidates["AcceptString"].str.startswith(text) |
        candidates["CityFA"].str.startswith(text)
    ):
        return result

    # 2) English city-name prefix
    if add_matches(
        candidates["City EN"].str.lower().str.startswith(text_lower)
    ):
        return result

    # 3) English keyboard used instead of Persian
    if add_matches(
        candidates["CityFA"].str.startswith(text_fa)
    ):
        return result

    # 4) Persian keyboard used instead of English
    if add_matches(
        candidates["City EN"].str.lower().str.startswith(text_en)
    ):
        return result

    # 5) Fuzzy typo matching with Edit Distance
    temp = candidates.copy()

    temp["distance"] = temp.apply(
        lambda row: min(
            distance(text, row["CityFA"]),
            distance(text_lower, row["City EN"].lower()),
            distance(text_fa, row["CityFA"]),
            distance(text_en, row["City EN"].lower())
        ),
        axis=1
    )

    temp = temp.sort_values(
        ["distance", "freq"],
        ascending=[True, False]
    )

    for value in temp["AcceptString"]:
        if value not in result:
            result.append(value)

        if len(result) == k:
            break

    return result


## 8. Generate the submission

`test_data.json` contains a `Typed` column. Five ranked, non-duplicate
suggestions are generated for every row.


In [ ]:
test_data = pd.read_json("test_data.json")

predictions = test_data["Typed"].apply(suggest)

submission = pd.DataFrame(
    predictions.tolist(),
    columns=[
        "Suggestion0",
        "Suggestion1",
        "Suggestion2",
        "Suggestion3",
        "Suggestion4"
    ]
)

submission.to_csv("submission.csv", index=False)
submission.head()


## Benchmark

The original submitted solution achieved **Mean RBO = 60.53** on the Quera
evaluation, while the stated full-score threshold was **30**.

RBO (Rank-Biased Overlap) rewards both overlap and ranking position, making it
appropriate for ordered suggestion lists.
